# ASL Baseline – Multi-modal Fusion with Co-occurrence Graph

**Changes vs original:**
- `build_dataloaders` now also returns `y_train` (needed to build co-occurrence matrix, no data leakage)
- `FusionConfig` gains `cooc_threshold` (default 0.2) and `cooc_embed_dim` (default 128)
- New class `LabelGraphConv` – 2-layer GCN that produces co-occurrence-aware label embeddings
- New model `CoOcGatedFusion` – identical to `GatedFusion` but adds a label-graph embedding branch whose scores are fused with the direct classifier logits
- All original models and training loops are **unchanged**

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
import os
import random
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, average_precision_score

warnings.filterwarnings("ignore")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from zipfile import ZipFile

file_name = "/content/drive/MyDrive/760_Project/dataset.zip"
if file_name:
    try:
        with ZipFile(file_name, 'r') as zip_obj:
            zip_obj.extractall("/content/")
        print(f"All contents extracted from {file_name}")
    except FileNotFoundError:
        print(f"Error: Zip file '{file_name}' not found.")
    except Exception as e:
        print(f"An error occurred during extraction: {e}")
else:
    print("Please provide a valid zip file name for 'file_name'.")

All contents extracted from /content/drive/MyDrive/760_Project/dataset.zip


In [ ]:
PROJECT_ROOT = Path(r"/content/dataset")
os.chdir(PROJECT_ROOT)

FEATURE_A_PATH = Path("Extracted_Features/BoW_int.npy")
FEATURE_B_PATHS = [
    Path("Extracted_Features/Normalized_CH.npy"),
    Path("Extracted_Features/Normalized_CM55.npy"),
    Path("Extracted_Features/Normalized_CORR.npy"),
    Path("Extracted_Features/Normalized_EDH.npy"),
    Path("Extracted_Features/Normalized_WT.npy"),
]
TEST_FEATURE_A_PATH = Path("Extracted_Features_Test/BoW_int.npy")
TEST_FEATURE_B_PATHS = [
    Path("Extracted_Features_Test/Normalized_CH.npy"),
    Path("Extracted_Features_Test/Normalized_CM55.npy"),
    Path("Extracted_Features_Test/Normalized_CORR.npy"),
    Path("Extracted_Features_Test/Normalized_EDH.npy"),
    Path("Extracted_Features_Test/Normalized_WT.npy"),
]

In [ ]:
COMMON_CONFIG = dict(
    batch_size=32,
    epochs=35,
    lr=1e-4,
    weight_decay=1e-4,
    hidden_dims=[1024, 512, 256],
    dropout=0.3,
    activation="gelu",
    loss="asl",
    asl_gamma_neg=4.0,
    asl_gamma_pos=1.0,
    asl_clip=0.05,
    threshold=0.5,
    top_k=5,
    val_ratio=0.2,
    random_seed=42,
    standardize=False,
)

In [ ]:
@dataclass
class FusionConfig:
    train_label_path: Path
    test_label_path: Path
    batch_size: int
    epochs: int
    lr: float
    weight_decay: float
    hidden_dims: list[int]
    dropout: float
    activation: str
    loss: str
    asl_gamma_neg: float
    asl_gamma_pos: float
    asl_clip: float
    threshold: float
    top_k: int
    val_ratio: float
    random_seed: int
    standardize: bool
    embed_dim: int = 128
    lambda_nce: float = 0.1
    temperature: float = 0.1
    device: str = "cuda"
    # ── Co-occurrence settings (NEW) ──────────────────────────────────────
    cooc_threshold: float = 0.2   # edges with P(j|i) < threshold are pruned
    cooc_embed_dim: int = 256     # label embedding dim in the GCN
    lambda_cooc: float = 0.5      # weight of the co-occurrence logits in fusion


config = FusionConfig(
    **COMMON_CONFIG,
    train_label_path=Path("database_labels_81_big.npy"),
    test_label_path=Path("database_labels_81_test.npy"),
)

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(config.random_seed)
DEVICE = torch.device(config.device if config.device == "cuda" and torch.cuda.is_available() else "cpu")

print(f"Project root : {Path.cwd()}")
print(f"Using device : {DEVICE}")

Project root : /content/dataset
Using device : cuda


In [ ]:
class MultiModalDataset(Dataset):
    def __init__(self, xa: np.ndarray, xb: np.ndarray, y: np.ndarray):
        self.xa = torch.tensor(xa, dtype=torch.float32)
        self.xb = torch.tensor(xb, dtype=torch.float32)
        self.y  = torch.tensor(y,  dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        return self.xa[idx], self.xb[idx], self.y[idx]

In [ ]:
def load_array(path: Path) -> np.ndarray:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path.resolve()}")
    return np.load(path).astype(np.float32)

def load_feature_group(path_or_paths: Path | list[Path]) -> np.ndarray:
    if isinstance(path_or_paths, Path):
        return load_array(path_or_paths)
    arrays = [load_array(p) for p in path_or_paths]
    return np.concatenate(arrays, axis=1)

def maybe_standardize(
    xa_train, xa_val, xa_test,
    xb_train, xb_val, xb_test,
    standardize: bool,
):
    if not standardize:
        return xa_train, xa_val, xa_test, xb_train, xb_val, xb_test
    scaler_a = StandardScaler().fit(xa_train)
    scaler_b = StandardScaler().fit(xb_train)
    return (
        scaler_a.transform(xa_train).astype(np.float32),
        scaler_a.transform(xa_val).astype(np.float32),
        scaler_a.transform(xa_test).astype(np.float32),
        scaler_b.transform(xb_train).astype(np.float32),
        scaler_b.transform(xb_val).astype(np.float32),
        scaler_b.transform(xb_test).astype(np.float32),
    )

In [ ]:
# NOTE: now returns y_train as well (used to build co-occurrence matrix)
def build_dataloaders(config: FusionConfig):
    xa_all   = load_feature_group(FEATURE_A_PATH)
    xb_all   = load_feature_group(FEATURE_B_PATHS)
    y_all    = load_array(config.train_label_path)
    xa_test  = load_feature_group(TEST_FEATURE_A_PATH)
    xb_test  = load_feature_group(TEST_FEATURE_B_PATHS)
    y_test   = load_array(config.test_label_path)

    if len(xa_all) != len(xb_all) or len(xa_all) != len(y_all):
        raise ValueError("Train feature and label sample counts do not match.")
    if len(xa_test) != len(xb_test) or len(xa_test) != len(y_test):
        raise ValueError("Test feature and label sample counts do not match.")

    idx_train, idx_val = train_test_split(
        np.arange(len(y_all)),
        test_size=config.val_ratio,
        random_state=config.random_seed,
        shuffle=True,
    )

    xa_train, xa_val = xa_all[idx_train], xa_all[idx_val]
    xb_train, xb_val = xb_all[idx_train], xb_all[idx_val]
    y_train,  y_val  = y_all[idx_train],  y_all[idx_val]

    xa_train, xa_val, xa_test, xb_train, xb_val, xb_test = maybe_standardize(
        xa_train, xa_val, xa_test, xb_train, xb_val, xb_test, config.standardize
    )

    train_loader = DataLoader(MultiModalDataset(xa_train, xb_train, y_train),
                              batch_size=config.batch_size, shuffle=True)
    val_loader   = DataLoader(MultiModalDataset(xa_val,   xb_val,   y_val),
                              batch_size=config.batch_size, shuffle=False)
    test_loader  = DataLoader(MultiModalDataset(xa_test,  xb_test,  y_test),
                              batch_size=config.batch_size, shuffle=False)

    print("Data loading completed.")
    print(f"Feature A (BoW) dim          : {xa_train.shape[1]}")
    print(f"Feature B (Color+Texture) dim: {xb_train.shape[1]}")
    print(f"Labels / classes             : {y_train.shape[1]}")
    print(f"Train: {len(xa_train)}  Val: {len(xa_val)}  Test: {len(xa_test)}")

    return (
        train_loader, val_loader, test_loader,
        xa_train.shape[1], xb_train.shape[1], y_train.shape[1],
        y_train,   # ← NEW: returned for co-occurrence matrix construction
    )


train_loader, val_loader, test_loader, DIM_A, DIM_B, NUM_CLASSES, Y_TRAIN = build_dataloaders(config)

Data loading completed.
Feature A (BoW) dim          : 500
Feature B (Color+Texture) dim: 634
Labels / classes             : 81
Train: 154987  Val: 38747  Test: 2100


## Co-occurrence Matrix  ← NEW

Built **once** from `Y_TRAIN` only (no leakage from val/test).

- `cooc[i,j] = P(label j | label i)` — conditional co-occurrence probability
- Thresholded at `config.cooc_threshold` (default 0.2) → binary adjacency
- Row-normalised: `A = D⁻¹ · A` for stable graph convolution

In [ ]:
def build_cooccurrence_matrix(label_matrix: np.ndarray, threshold: float = 0.2) -> torch.Tensor:
    """
    Parameters
    ----------
    label_matrix : (N, C) binary np.ndarray  ← Y_TRAIN
    threshold    : prune edges weaker than this conditional probability

    Returns
    -------
    A : (C, C) row-normalised adjacency tensor (float32)
    """
    # Raw counts: cooc[i,j] = # samples carrying BOTH label i and label j
    cooc = label_matrix.T @ label_matrix                       # (C, C)

    # Conditional probability: P(j | i)
    diag     = cooc.diagonal().clip(min=1)                     # avoid /0
    cooc_prob = cooc / diag[:, None]                           # (C, C)

    # Threshold → binary adjacency
    A = (cooc_prob > threshold).astype(np.float32)
    np.fill_diagonal(A, 1.0)                                   # self-loops
    A = np.maximum(A, A.T)                                     # symmetrise

    # Row-normalise: D⁻¹ A
    row_sums = A.sum(axis=1, keepdims=True).clip(min=1)
    A = A / row_sums

    return torch.from_numpy(A.astype(np.float32))


A_COOC = build_cooccurrence_matrix(Y_TRAIN, threshold=config.cooc_threshold)
print(f"Co-occurrence adjacency : {A_COOC.shape}")
print(f"Non-zero edges          : {(A_COOC > 0).sum().item()} / {NUM_CLASSES ** 2}")

Co-occurrence adjacency : torch.Size([81, 81])
Non-zero edges          : 593 / 6561


## Helpers (unchanged)

In [ ]:
def make_activation(name: str) -> nn.Module:
    activations = {"relu": nn.ReLU, "gelu": nn.GELU, "silu": nn.SiLU}
    if name not in activations:
        raise ValueError(f"Unsupported activation: {name}")
    return activations[name]()


def make_mlp(
    input_dim: int, hidden_dims: list[int], output_dim: int,
    dropout: float, activation: str,
):
    # 1. Tạo phần thân (trích xuất đặc trưng)
    layers: list[nn.Module] = []
    prev_dim = input_dim
    for h in hidden_dims:
        layers += [nn.Linear(prev_dim, h), make_activation(activation)]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        prev_dim = h

    backbone = nn.Sequential(*layers)

    # 2. Tạo lớp cuối cùng (classifier/regressor)
    head = nn.Linear(prev_dim, output_dim)

    return backbone, head


def info_nce_loss(h1: torch.Tensor, h2: torch.Tensor, temperature: float = 0.1) -> torch.Tensor:
    h1     = F.normalize(h1, dim=-1)
    h2     = F.normalize(h2, dim=-1)
    logits = torch.matmul(h1, h2.T) / temperature
    labels = torch.arange(h1.size(0), device=h1.device)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2.0

## Original Models (unchanged)

In [ ]:
class BaselineModel(nn.Module):
    def __init__(self, input_dim: int, feature_type: str, config: FusionConfig):
        super().__init__()
        self.feature_type = feature_type
        self.backbone, self.classifier = make_mlp(input_dim, config.hidden_dims, NUM_CLASSES,
                            config.dropout, config.activation)

    def forward(self, xa: torch.Tensor, xb: torch.Tensor):
        x = xa if self.feature_type == "A" else xb
        return self.net(x), None, None


class EarlyFusion(nn.Module):
    def __init__(self, config: FusionConfig):
        super().__init__()
        self.proj_A     = nn.Sequential(nn.Linear(DIM_A, config.embed_dim), make_activation(config.activation))
        self.proj_B     = nn.Sequential(nn.Linear(DIM_B, config.embed_dim), make_activation(config.activation))
        self.backbone, self.classifier = make_mlp(config.embed_dim * 2, config.hidden_dims, NUM_CLASSES,
                                   config.dropout, config.activation)

    def forward(self, xa: torch.Tensor, xb: torch.Tensor):
        h1 = self.proj_A(xa)
        h2 = self.proj_B(xb)
        return self.classifier(self.backbone(torch.cat([h1, h2], dim=1))), h1, h2

class EarlyFusion_Raw(nn.Module):
    def __init__(self, config: FusionConfig):
        super().__init__()
        self.proj_A     = nn.Sequential(nn.Linear(DIM_A, config.embed_dim), make_activation(config.activation))
        self.proj_B     = nn.Sequential(nn.Linear(DIM_B, config.embed_dim), make_activation(config.activation))
        self.backbone, self.classifier = make_mlp(1134, config.hidden_dims, NUM_CLASSES,
                                   config.dropout, config.activation)

    def forward(self, xa: torch.Tensor, xb: torch.Tensor):
        feature_h = torch.cat([xa, xb], dim=1)
        return self.classifier(self.backbone(feature_h)), xa, xb


class GatedFusion(nn.Module):
    def __init__(self, config: FusionConfig):
        super().__init__()
        self.proj_A     = nn.Sequential(nn.Linear(DIM_A, config.embed_dim), make_activation(config.activation))
        self.proj_B     = nn.Sequential(nn.Linear(DIM_B, config.embed_dim), make_activation(config.activation))
        self.gate_layer = nn.Linear(config.embed_dim * 2, config.embed_dim)
        self.classifier = make_mlp(config.embed_dim, config.hidden_dims, NUM_CLASSES,
                                   config.dropout, config.activation)

    def forward(self, xa: torch.Tensor, xb: torch.Tensor):
        h1    = self.proj_A(xa)
        h2    = self.proj_B(xb)
        gate  = torch.sigmoid(self.gate_layer(torch.cat([h1, h2], dim=-1)))
        fused = gate * h1 + (1.0 - gate) * h2
        return self.classifier(fused), h1, h2

## Label Graph Convolution  ← NEW

```
Label embeddings (C × embed_dim)
        │
  GC-1: A · X · W₁  → LeakyReLU
        │
  GC-2: A · X · W₂  → LeakyReLU
        │
  Z : (C, cooc_embed_dim)   ← co-occurrence-aware label vectors
```

The adjacency matrix `A` is registered as a **buffer** (fixed, never updated by the optimizer).

In [ ]:
class LabelGraphConv(nn.Module):
    """
    2-layer GCN that converts learnable label embeddings into
    co-occurrence-aware representations.

    Parameters
    ----------
    num_labels  : C  (number of classes)
    embed_dim   : dimension of each label vector
    """
    def __init__(self, num_labels: int, embed_dim: int = 128):
        super().__init__()
        self.label_emb = nn.Embedding(num_labels, embed_dim)
        self.gc1 = nn.Linear(embed_dim, embed_dim, bias=False)
        self.gc2 = nn.Linear(embed_dim, embed_dim, bias=False)
        self.act = nn.LeakyReLU(0.2, inplace=True)

        nn.init.xavier_uniform_(self.label_emb.weight)
        nn.init.xavier_uniform_(self.gc1.weight)
        nn.init.xavier_uniform_(self.gc2.weight)

    def forward(self, A: torch.Tensor) -> torch.Tensor:
        """
        A : (C, C) row-normalised adjacency on the correct device
        Returns Z : (C, embed_dim)
        """
        idx = torch.arange(self.label_emb.num_embeddings, device=A.device)
        X = self.label_emb(idx)           # (C, embed_dim)
        X = self.act(self.gc1(A @ X))     # 1st graph conv
        Z = self.act(self.gc2(A @ X))     # 2nd graph conv
        return Z                          # (C, embed_dim)

## CoOcGatedFusion  ← NEW

Architecture:
```
xa ──► proj_A ──┐
                ├─► gate ──► fused (B, embed_dim)
xb ──► proj_B ──┘
                │
         ┌──────┴──────────────────────────────────────┐
         │  Branch 1 (direct)    │  Branch 2 (co-occ)  │
         │  MLP → logits_cls     │  feat_proj + GCN    │
         │  (C,)                 │  dot-product → (C,) │
         └──────────────────────────────────────────────┘
                         ↓
         logits = logits_cls + λ · logits_cooc
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleLabelCorrelation(nn.Module):
    def __init__(self, config: FusionConfig, adjacency: torch.Tensor):
        super().__init__()

        # 1. Projection layers cho 2 nguồn dữ liệu đầu vào (A và B)
        # Sử dụng config.embed_dim từ FusionConfig
        self.proj_A = nn.Sequential(
            nn.Linear(DIM_A, config.embed_dim),
            make_activation(config.activation)
        )
        self.proj_B = nn.Sequential(
            nn.Linear(DIM_B, config.embed_dim),
            make_activation(config.activation)
        )

        # 2. MLP Backbone và Head dự đoán độc lập
        # Sử dụng config.hidden_dims, NUM_CLASSES, config.dropout từ FusionConfig
        self.backbone, self.classifier_head = make_mlp(
            input_dim=config.embed_dim * 2,
            hidden_dims=config.hidden_dims,
            output_dim=NUM_CLASSES,
            dropout=config.dropout,
            activation=config.activation
        )

        # 3. Lưu trữ ma trận kề cố định vào buffer
        self.register_buffer("A", adjacency) # (NUM_CLASSES, NUM_CLASSES)

        # 4. Trọng số học được cho Label Correlation
        # Khởi tạo bằng giá trị lambda_cooc từ config của bạn
        self.lambda_cooc = nn.Parameter(torch.tensor(config.lambda_cooc))

    def forward(self, xa: torch.Tensor, xb: torch.Tensor):
        # Bước 1: Feature Projection
        h1 = self.proj_A(xa)
        h2 = self.proj_B(xb)

        # Bước 2: Feature Fusion qua concat và backbone
        combined = torch.cat([h1, h2], dim=1)
        feat = self.backbone(combined)

        # Bước 3: Đưa ra dự đoán thô (Independent Logits)
        logits_raw = self.classifier_head(feat) # (Batch, NUM_CLASSES)

        # Bước 4: Áp dụng Simple Label Correlation qua ma trận kề
        # Logic: Score của một nhãn sẽ được bổ sung bởi score của các nhãn liên quan
        logits_correlated = torch.matmul(logits_raw, self.A)

        # Bước 5: Fuse kết quả dựa trên lambda_cooc
        # final_logits = y_raw + lambda * (y_raw @ A)
        final_logits = logits_raw + self.lambda_cooc * logits_correlated

        # Trả về logits và h1, h2 để bạn có thể tính thêm Contrastive Loss (InfoNCE) bên ngoài
        return final_logits, h1, h2

In [ ]:
class CoO_GCN(nn.Module):
    def __init__(self, config: FusionConfig, adjacency: torch.Tensor):
        super().__init__()

        # 1. Projection layers cho 2 nguồn dữ liệu đầu vào (A và B)
        # Sử dụng config.embed_dim từ FusionConfig
        self.proj_A = nn.Sequential(
            nn.Linear(DIM_A, config.embed_dim),
            make_activation(config.activation)
        )
        self.proj_B = nn.Sequential(
            nn.Linear(DIM_B, config.embed_dim),
            make_activation(config.activation)
        )

        # 2. MLP Backbone và Head dự đoán độc lập
        # Sử dụng config.hidden_dims, NUM_CLASSES, config.dropout từ FusionConfig
        self.backbone, self.classifier_head = make_mlp(
            input_dim=config.embed_dim * 2,
            hidden_dims=config.hidden_dims,
            output_dim=NUM_CLASSES,
            dropout=config.dropout,
            activation=config.activation
        )

        # 3. Nhánh GCN để tạo ra nhãn nhúng (Label Embeddings)
        self.label_gcn = LabelGraphConv(NUM_CLASSES, config.cooc_embed_dim)

        # 4. Ma trận kề lưu trong buffer
        self.register_buffer("A", adjacency)

        # 5. Temperature parameter: Cực kỳ quan trọng khi dùng Cosine Similarity
        # Nếu để giá trị thấp (0.07), mô hình sẽ phân biệt các nhãn gắt hơn
        self.temperature = config.temperature

    def forward(self, xa: torch.Tensor, xb: torch.Tensor):
        # 1. Feature Extraction
        h1 = self.proj_A(xa)
        h2 = self.proj_B(xb)
        combined = torch.cat([h1, h2], dim=1)

        # Lấy đặc trưng từ backbone (trước khi qua head)
        feat = self.backbone(combined)

        # --- NHÁNH 1: Classification Bình thường (Independent) ---
        # Sử dụng classifier_head (nn.Linear)
        logits_mlp = self.classifier_head(feat)

        # --- NHÁNH 2: Dynamic Classification (GCN-based) ---
        feat_normalized = F.normalize(feat, p=2, dim=-1)
        label_emb = F.normalize(self.label_gcn(self.A), p=2, dim=-1)

        # Tính cosine similarity và chia temperature
        logits_dynamic = torch.matmul(feat_normalized, label_emb.t()) / self.temperature

        # --- KẾT HỢP (HYBRID) ---
        # Sử dụng lambda_cooc để điều tiết (ví dụ 0.5 là tin tưởng cả hai như nhau)
        final_logits = logits_mlp + self.lambda_cooc * logits_dynamic

        return final_logits, h1, h2

In [ ]:
class CoOcGatedFusion(nn.Module):
    """
    GatedFusion extended with a co-occurrence label graph branch.

    The co-occurrence adjacency `A` is passed in at construction time and
    stored as a non-trainable buffer so it moves to the correct device
    automatically with `.to(device)`.

    Forward returns the same tuple as all other models:
        (logits, h1, h2)
    so it is 100 % compatible with the existing train / evaluate loops.
    """
    def __init__(self, config: FusionConfig, adjacency: torch.Tensor):
        super().__init__()

        # ── Fixed co-occurrence graph ─────────────────────────────────────
        self.register_buffer("A", adjacency)          # (C, C)

        # ── Shared gated feature fusion (identical to GatedFusion) ────────
        self.proj_A     = nn.Sequential(
            nn.Linear(DIM_A, config.embed_dim), make_activation(config.activation)
        )
        self.proj_B     = nn.Sequential(
            nn.Linear(DIM_B, config.embed_dim), make_activation(config.activation)
        )
        self.gate_layer = nn.Linear(config.embed_dim * 2, config.embed_dim)

        # ── Branch 1: direct MLP classifier (same as GatedFusion) ─────────
        self.classifier = make_mlp(
            config.embed_dim, config.hidden_dims, NUM_CLASSES,
            config.dropout, config.activation,
        )

        # ── Branch 2: co-occurrence label-graph embedding ← NEW ───────────
        # Projects fused image features into the same space as label vectors
        self.feat_proj = nn.Sequential(
            nn.Linear(config.embed_dim, config.cooc_embed_dim),
            make_activation(config.activation),
        )
        self.label_gcn = LabelGraphConv(NUM_CLASSES, config.cooc_embed_dim)

        # Learnable fusion scalar (initialised to lambda_cooc from config)
        self.lambda_cooc = nn.Parameter(torch.tensor(config.lambda_cooc))

    # ─────────────────────────────────────────────────────────────────────
    def forward(self, xa: torch.Tensor, xb: torch.Tensor):
        # 1. Gated feature fusion
        h1    = self.proj_A(xa)                                # (B, embed_dim)
        h2    = self.proj_B(xb)                                # (B, embed_dim)
        gate  = torch.sigmoid(self.gate_layer(torch.cat([h1, h2], dim=-1)))
        fused = gate * h1 + (1.0 - gate) * h2                 # (B, embed_dim)

        # 2. Branch 1 – direct classifier
        logits_cls  = self.classifier(fused)                   # (B, C)

        # 3. Branch 2 – co-occurrence label-graph branch
        feat_emb    = F.normalize(self.feat_proj(fused), dim=-1)   # (B, cooc_embed_dim)
        label_emb   = F.normalize(self.label_gcn(self.A),   dim=-1) # (C, cooc_embed_dim)
        logits_cooc = feat_emb @ label_emb.t()                     # (B, C)

        # 4. Fuse both branches
        logits = logits_cls + self.lambda_cooc * logits_cooc  # (B, C)

        return logits, h1, h2

## Loss & Evaluation (unchanged)

In [ ]:
class AsymmetricLossWithLogits(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05, eps=1e-8, reduction="mean"):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip      = clip
        self.eps       = eps
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs     = torch.sigmoid(logits)
        pos_probs = probs
        neg_probs = 1.0 - probs
        if self.clip is not None and self.clip > 0:
            neg_probs = (neg_probs + self.clip).clamp(max=1.0)
        pos_loss   = targets       * torch.log(pos_probs.clamp(min=self.eps))
        neg_loss   = (1 - targets) * torch.log(neg_probs.clamp(min=self.eps))
        pos_weight = torch.pow(1.0 - pos_probs, self.gamma_pos)
        neg_weight = torch.pow(1.0 - neg_probs, self.gamma_neg)
        loss       = -(pos_weight * pos_loss + neg_weight * neg_loss)
        if self.reduction == "mean": return loss.mean()
        if self.reduction == "sum":  return loss.sum()
        return loss


def evaluate(model: nn.Module, loader: DataLoader, threshold: float = 0.5):
    model.eval()
    all_targets, all_probs, all_preds = [], [], []
    with torch.no_grad():
        for xa, xb, y in loader:
            xa, xb, y = xa.to(DEVICE), xb.to(DEVICE), y.to(DEVICE)
            logits, _, _ = model(xa, xb)
            probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu().numpy())
            all_preds.append((probs > threshold).float().cpu().numpy())
            all_targets.append(y.cpu().numpy())
    all_probs   = np.vstack(all_probs)
    all_preds   = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    mi_f1 = f1_score(all_targets, all_preds, average="micro", zero_division=0)
    ma_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    try:    map_score = average_precision_score(all_targets, all_probs, average="macro")
    except: map_score = 0.0
    return map_score, mi_f1, ma_f1


def build_criterion(config: FusionConfig) -> nn.Module:
    if config.loss == "bce": return nn.BCEWithLogitsLoss()
    if config.loss == "asl":
        return AsymmetricLossWithLogits(
            gamma_neg=config.asl_gamma_neg,
            gamma_pos=config.asl_gamma_pos,
            clip=config.asl_clip,
        )
    raise ValueError(f"Unsupported loss: {config.loss}")

## Training loop (unchanged)

In [ ]:
def train_model(model_name: str, model: nn.Module, config: FusionConfig, use_infonce: bool = False):
    model     = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    criterion = build_criterion(config)

    print(f"\n{'=' * 55}\nTraining: {model_name}\n{'=' * 55}")
    print(f"hidden_dims={config.hidden_dims}  activation={config.activation}  dropout={config.dropout}")

    for epoch in range(config.epochs):
        model.train()
        epoch_loss = 0.0
        for xa, xb, y in train_loader:
            xa, xb, y = xa.to(DEVICE), xb.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits, h1, h2 = model(xa, xb)
            loss = criterion(logits, y)
            if use_infonce and h1 is not None and h2 is not None:
                loss = loss + config.lambda_nce * info_nce_loss(h1, h2, config.temperature)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        val_map, val_mi, val_ma = evaluate(model, val_loader, config.threshold)
        print(
            f"Epoch [{epoch+1:02d}/{config.epochs}] "
            f"Loss: {epoch_loss / max(len(train_loader),1):.4f} | "
            f"Val mAP: {val_map:.4f} | Mi-F1: {val_mi:.4f} | Ma-F1: {val_ma:.4f}"
        )

    test_map, test_mi, test_ma = evaluate(model, test_loader, config.threshold)
    print(f"[TEST] mAP: {test_map:.4f} | Micro-F1: {test_mi:.4f} | Macro-F1: {test_ma:.4f}")
    return {"mAP": test_map, "Mi_F1": test_mi, "Ma_F1": test_ma, "model": model}

## Experiments

In [ ]:
results = {}

# ── Original baselines (unchanged) ───────────────────────────────────────────
# results["Baseline_A (BoW)"] = train_model(
#     "Baseline (Local: BoW)", BaselineModel(DIM_A, "A", config), config)

# results["Baseline_B (Color+Tex)"] = train_model(
#     "Baseline (Global: Color+Texture)", BaselineModel(DIM_B, "B", config), config)

# results["Early_Fusion (Concat)"] = train_model(
#     "Early Fusion (Concat)", EarlyFusion(config), config)

# results["Early_Fusion (Raw)"] = train_model(
#     "Early Fusion (Raw)", EarlyFusion_Raw(config), config)

# results["Early_Fusion (Concate)"] = train_model(
#     "Early_Fusion (Concate) + Co-occurence", CoOc_Concate(config, A_COOC), config)

results["Early_Fusion (Concate)"] = train_model(
    "Early_Fusion (Concate) + Co-occurence", SimpleLabelCorrelation(config, A_COOC), config)


# results["Gated_Fusion + InfoNCE"] = train_model(
#     "Gated Fusion + InfoNCE", GatedFusion(config), config, use_infonce=True)

# results["Gated_Fusion (No NCE)"] = train_model(
#     "Gated Fusion (No InfoNCE)", GatedFusion(config), config, use_infonce=False)

# ── NEW: Co-occurrence Gated Fusion (no InfoNCE) ──────────────────────────────
# results["CoOc_GatedFusion"] = train_model(
#     "CoOc Gated Fusion",
#     CoOcGatedFusion(config, adjacency=A_COOC),   # ← pass co-occurrence graph
#     config,
#     use_infonce=False,
# )

# # ── NEW: Co-occurrence Gated Fusion + InfoNCE ─────────────────────────────────
# results["CoOc_GatedFusion + InfoNCE"] = train_model(
#     "CoOc Gated Fusion + InfoNCE",
#     CoOcGatedFusion(config, adjacency=A_COOC),
#     config,
#     use_infonce=True,
# )

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "=" * 68)
print(f"{'Experiment Model':<35} | {'mAP (%)':<8} | {'Micro-F1':<8} | {'Macro-F1'}")
print("-" * 68)
for name, res in results.items():
    print(f"{name:<35} | {res['mAP']*100:.2f}     | {res['Mi_F1']*100:.2f}     | {res['Ma_F1']*100:.2f}")
print("=" * 68)


Training: Early_Fusion (Concate) + Co-occurence
hidden_dims=[1024, 512, 256]  activation=gelu  dropout=0.3
Epoch [01/35] Loss: 0.0138 | Val mAP: 0.2051 | Mi-F1: 0.5431 | Ma-F1: 0.1854
Epoch [02/35] Loss: 0.0120 | Val mAP: 0.2357 | Mi-F1: 0.5585 | Ma-F1: 0.2207
Epoch [03/35] Loss: 0.0116 | Val mAP: 0.2562 | Mi-F1: 0.5662 | Ma-F1: 0.2419
Epoch [04/35] Loss: 0.0113 | Val mAP: 0.2670 | Mi-F1: 0.5732 | Ma-F1: 0.2561
Epoch [05/35] Loss: 0.0110 | Val mAP: 0.2773 | Mi-F1: 0.5747 | Ma-F1: 0.2709
Epoch [06/35] Loss: 0.0109 | Val mAP: 0.2837 | Mi-F1: 0.5769 | Ma-F1: 0.2747
Epoch [07/35] Loss: 0.0107 | Val mAP: 0.2893 | Mi-F1: 0.5762 | Ma-F1: 0.2850
Epoch [08/35] Loss: 0.0106 | Val mAP: 0.2934 | Mi-F1: 0.5760 | Ma-F1: 0.2944
Epoch [09/35] Loss: 0.0104 | Val mAP: 0.2959 | Mi-F1: 0.5817 | Ma-F1: 0.2970
Epoch [10/35] Loss: 0.0103 | Val mAP: 0.2979 | Mi-F1: 0.5824 | Ma-F1: 0.3022
Epoch [11/35] Loss: 0.0102 | Val mAP: 0.3019 | Mi-F1: 0.5839 | Ma-F1: 0.3088
Epoch [12/35] Loss: 0.0101 | Val mAP: 0.3056 

[TEST] mAP: 0.3485 | Micro-F1: 0.6277 | Macro-F1: 0.3127

Early_Fusion (Raw)                  | 30.90     | 61.69     | 30.19

Early_Fusion (Concate) + CO              | 35.61     | 63.12     | 34.55

## t-SNE Visualisation (unchanged)

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def visualize_embeddings(model, dataloader, num_samples=500):
    model.eval()
    model.to(DEVICE)
    h1_list, h2_list = [], []
    with torch.no_grad():
        for i, (xa, xb, _) in enumerate(dataloader):
            xa, xb = xa.to(DEVICE), xb.to(DEVICE)
            _, h1, h2 = model(xa, xb)
            if h1 is not None and h2 is not None:
                h1_list.append(h1.cpu().numpy())
                h2_list.append(h2.cpu().numpy())
            if sum(len(h) for h in h1_list) >= num_samples:
                break
    if not h1_list:
        print("Model has no multimodal hidden outputs (h1, h2) – visualisation skipped.")
        return
    h1_arr   = np.vstack(h1_list)[:num_samples]
    h2_arr   = np.vstack(h2_list)[:num_samples]
    combined = np.vstack([h1_arr, h2_arr])
    labels   = np.array([0]*num_samples + [1]*num_samples)
    print("Running t-SNE …")
    reduced = TSNE(n_components=2, random_state=42).fit_transform(combined)
    plt.figure(figsize=(10, 7))
    plt.scatter(reduced[labels==0, 0], reduced[labels==0, 1], alpha=0.6,
                label="Feature A (BoW)", c='#1f77b4', s=15)
    plt.scatter(reduced[labels==1, 0], reduced[labels==1, 1], alpha=0.6,
                label="Feature B (Color+Texture)", c='#d62728', s=15)
    plt.title("t-SNE Latent Space: Feature A vs Feature B", fontsize=14, fontweight='bold')
    plt.xlabel("t-SNE Dim 1"); plt.ylabel("t-SNE Dim 2")
    plt.legend(fontsize=11); plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout(); plt.show()


# Visualise original Gated + InfoNCE
print("\n Gated Fusion + InfoNCE latent space …")
if 'Gated_Fusion + InfoNCE' in results:
    visualize_embeddings(results['Gated_Fusion + InfoNCE']['model'], test_loader, num_samples=600)

# Visualise new CoOc model
print("\n CoOc Gated Fusion + InfoNCE latent space …")
if 'CoOc_GatedFusion + InfoNCE' in results:
    visualize_embeddings(results['CoOc_GatedFusion + InfoNCE']['model'], test_loader, num_samples=600)